# 67. Quantized Inference and Deployment | 量化推理与部署

**难度：** Hard | **环境：** [CPU 主线，GPU 可选](#runtime-environment) | **标签：** `量化压缩`、`配对部署` | **目标人群：** 掌握推理指标与量化基础的读者

## 本节导读

量化权重减少了存储字节，实际请求却可能变慢：加载器需要恢复数值，计算内核也可能更换，收益还受批量和生成长度影响。要决定是否保留量化部署，先让它与同源浮点模型处理同一组请求。

从已检查文件和质量的产物出发，分别运行浮点基线与量化候选，核对实际执行、质量及资源变化。配对通过后，再扩大请求压力，判断哪些服务场景适合使用这个候选。


<a id="runtime-environment"></a>

## 运行环境

- **CPU 练习：** 使用 [BASE-CPU](../docs/guide.md#base-cpu)，核对配对条件、比值方向和部署状态。
- **真实实验：** 使用 [INF-VLLM-011-CU128](../topic_discussion/quantization_compression/quickstart.md#q-vllm-quickstart)，需要兼容 GPU 与已准备的量化产物。
- **执行阶段：** `prepare` 检查来源，`measure` 测量并保存，`close` 读取记录并生成报告；准备与收尾阶段不启动后端。


## 前置阅读

先核对产物来源与加载结果，再比较同一模型的服务表现。

- [量化产物评估](./82_Quantization_Artifact_Evaluation_Project.ipynb)：提供本项目必需的产物引用。
- [推理性能对比](./66_Inference_Performance_Comparison.ipynb)：请求事件和服务指标；浮点基线也可在本项目重新测量。


## Step 1：让同源浮点模型与量化产物处理同一请求

先检查量化产物与基座模型的版本关系，再固定分词、请求内容、并发、生成规则和运行环境。浮点与量化服务依次运行，使单卡在每次测量时只驻留当前模型；多轮交换先后顺序，减少固定执行顺序对结果的影响。

加载完成、产生输出和实际使用低精度计算分别记录。保存每组请求、后端日志及失败原因后，再联合比较质量与资源条件，决定是否扩大负载复测。

| 阶段 | 比较对象 | 记录内容 |
| --- | --- | --- |
| 读取产物 | 量化文件与同源基座 | 文件摘要、版本及已有质量记录 |
| 浮点运行 | 原始模型与固定请求 | 请求输出、时间和运行条件 |
| 量化运行 | 量化产物与相同请求 | 实际加载和执行路径、时间与失败 |
| 联合判断 | 多轮配对与质量记录 | 接受、调整、拒绝或待补证据 |

![同一输入下的浮点与量化部署比较](../docs/public/02_PyTorch_Algorithms/67_quantized_deployment_flow.svg)


## Step 2：固定配对实验的可比条件

配对实验只有在模型语义、请求压力和运行环境一致时才能解释收益。浮点基线与量化候选固定模型和分词器版本、请求负载、执行后端、硬件与缓存策略，只改变权重产物和对应的低精度执行路径。

| 可比条件 | G0 浮点基线 | G1 量化候选 | 判断方式 |
|:---|:---|:---|:---|
| 模型身份 | 基座模型与不可变版本 | 来自同一基座模型 | 模型版本与分词器版本一致 |
| 文本处理 | 固定分词器版本 | 与 G0 相同 | 编码规则与特殊 Token 一致 |
| 请求负载 | 固定提示、请求数、并发与生成上限 | 与 G0 相同 | 使用同一份已版本化的请求集 |
| 运行环境 | 固定后端、GPU 与缓存策略 | 与 G0 相同 | 环境快照一致 |
| 执行对象 | 浮点模型 | 82 已准入的量化产物 | 仅产物和量化启动路径改变 |


## Step 3：执行证据、质量与资源条件怎样共同决定去向

先核对两组是否可比，再确认真实后端执行了量化路径且未回退。质量指标需要说明方向：本题和默认项目采用负对数似然（NLL）一类越低越好的指标，候选与基线的比值越大表示退化越多；准确率等越高越好的指标要先换用对应判断规则。

当前筛选规则要求质量通过，并至少满足延迟、吞吐或显存中的一项条件。这些条件可以容许一定性能回退，例如 TPOT 比不高于 1.20、吞吐比不低于 0.90；达到它们表示进入后续复测的条件成立。是否真正加速，还需读取比值相对 1 的方向和所有服务约束，记录为当前负载下的具体收益与代价。

| 检查对象 | 条件或指标方向 | 当前候选的处理 |
| --- | --- | --- |
| 配对身份 | 固定字段一致、来自已接受产物 | 错配则拒绝并修复 |
| 实际执行 | 后端、量化路径、无回退三项同时成立 | 缺证则保持待验证 |
| 质量 | 越低越好指标的候选／基线比不超过上限 | 超限则拒绝并调整量化产物 |
| 延迟与吞吐 | TPOT 比按上限，吞吐比按下限 | 与显存条件一起作“至少一项满足”筛选 |
| 显存 | 相对降幅达到门槛 | 缺测保留为空，继续检查其余条件 |

![不同证据如何进入部署判断](../docs/public/02_PyTorch_Algorithms/67_quantization_evidence_flow.svg)


## Step 4：实现配对条件、指标换算与部署状态

本题提供产物来源、固定字段比较和原始指标，骨架负责提取检查事实及报告结构。请补全配对条件、不同指标的比值与降幅，以及执行、质量和资源条件的组合。测试涵盖版本或硬件变化、缺少显存数据与四种决策状态。

| 任务与目的 | 已知条件 | 完成标准 | 测试验证 |
| --- | --- | --- | --- |
| TODO 1：核对来源与固定条件 | 接受状态、产物和模型版本、字段差异 | 来源通过且两组固定条件一致 | 硬件变化与模型版本变化 |
| TODO 2：换算配对指标 | 两组延迟、吞吐、显存和质量 | 计算候选／基线比及显存降幅 | 比值方向和显存缺测 |
| TODO 3：形成部署状态 | 执行事实、质量和资源门槛 | 执行与质量均通过，资源条件至少满足一项 | 待验证、拒绝、调整与接受 |


In [ ]:
from typing import Any, Dict

from tools.quantization_project_runtime import _finite_number, _positive_int, _text


In [ ]:
FIXED_PAIR_FIELDS = (
    "model_revision", "tokenizer_revision", "workload_id",
    "backend", "hardware_id", "cache_policy",
)


def _positive_ratio(candidate_value: float, baseline_value: float) -> float:
    """计算正数指标的候选/基线比值。"""
    if not _finite_number(baseline_value, positive=True) or not _finite_number(candidate_value) or candidate_value < 0:
        raise ValueError("指标必须满足 baseline > 0 且 candidate >= 0")
    return candidate_value / baseline_value


def validate_deployment_pair(
    gate: Dict[str, Any], baseline: Dict[str, Any], candidate: Dict[str, Any]
) -> Dict[str, Any]:
    """检查产物门禁与 G0/G1 是否构成可比较的部署配对。"""
    # 给定实现：提取门禁、产物血缘和固定字段漂移等原子事实。
    gate_ready = gate.get("decision") == "accept"
    artifact_matches = _text(candidate.get("artifact_id")) and candidate.get("artifact_id") == gate.get("artifact_id")
    model_revision_matches = _text(candidate.get("model_revision")) and candidate.get("model_revision") == gate.get("model_revision")
    fixed_mismatches = [
        field for field in FIXED_PAIR_FIELDS if not _text(baseline.get(field)) or baseline.get(field) != candidate.get(field)
    ]
    # TODO 1（项目机制：配对契约）：组合血缘门禁与固定条件。
    # lineage_ready = ???  # 82 已接受，且产物 ID、模型 revision 与 G1 一致。
    # fixed_fields_ready = ???  # G0/G1 的固定字段不能发生漂移。
    if not all(name in locals() for name in ("lineage_ready", "fixed_fields_ready")):
        raise NotImplementedError("TODO 1：请完成血缘门禁与固定条件判断")
    issues = []
    if not gate_ready:
        issues.append("artifact_gate_not_accepted")
    if not artifact_matches:
        issues.append("artifact_id_mismatch")
    if not model_revision_matches:
        issues.append("model_revision_mismatch")
    issues.extend(f"fixed_field_mismatch:{field}" for field in fixed_mismatches)
    return {
        "ready": lineage_ready and fixed_fields_ready,
        "issues": issues,
        "fixed_mismatches": fixed_mismatches,
    }


def summarize_deployment_pair(
    baseline: Dict[str, Any], candidate: Dict[str, Any]
) -> Dict[str, Any]:
    """把延迟、吞吐、显存和质量转换成方向明确的配对指标。"""
    # TODO 2（项目机制：指标账本）：比值小于 1 的 TPOT 更好，比值大于 1 的吞吐更好。
    # tpot_ratio = ???  # candidate tpot / baseline tpot。
    # throughput_ratio = ???  # candidate throughput / baseline throughput。
    # memory_reduction_ratio = ???  # 显存缺失时保留 None。
    # quality_ratio = ???  # candidate quality / baseline quality。
    if not all(name in locals() for name in ("tpot_ratio", "throughput_ratio", "memory_reduction_ratio", "quality_ratio")):
        raise NotImplementedError("TODO 2：请完成四项配对指标")
    return {
        "tpot_ratio": round(tpot_ratio, 6),
        "throughput_ratio": round(throughput_ratio, 6),
        "memory_reduction_ratio": None if memory_reduction_ratio is None else round(memory_reduction_ratio, 6),
        "quality_ratio": round(quality_ratio, 6),
    }


def decide_quantized_deployment(
    contract: Dict[str, Any], comparison: Dict[str, Any], evidence: Dict[str, Any],
    *, max_tpot_ratio: float = 1.20, min_throughput_ratio: float = 0.90,
    min_memory_reduction_ratio: float = 0.20, max_quality_ratio: float = 1.10,
) -> Dict[str, str]:
    """根据配对结果和真实执行证据形成阶段化部署决策。"""
    if not all(_finite_number(v) and v >= 0 for v in (max_tpot_ratio, min_throughput_ratio, min_memory_reduction_ratio, max_quality_ratio)):
        raise ValueError('部署门槛必须是有限非负数')
    if any(not _finite_number(comparison.get(k)) or comparison[k] < 0 for k in ('tpot_ratio', 'throughput_ratio', 'quality_ratio')):
        return {'decision': 'pending', 'reason': '部署比较缺少有效数值'}
    memory_gain = comparison.get("memory_reduction_ratio")
    # 给定实现：从真实记录中提取执行路径与三类资源目标。
    backend_observed = evidence.get("real_backend") is True
    quantized_path_observed = evidence.get("quantized_path") is True
    no_fallback = evidence.get("fallback") is False
    latency_ready = comparison.get("tpot_ratio", float("inf")) <= max_tpot_ratio
    throughput_ready = comparison.get("throughput_ratio", 0.0) >= min_throughput_ratio
    memory_ready = memory_gain is not None and memory_gain >= min_memory_reduction_ratio
    # TODO 3（项目机制：部署状态机）：组合执行、质量和资源门槛。
    # evidence_ready = ???  # 后端、量化路径和无回退证据必须同时成立。
    # quality_ready = ???  # 质量比值不能超过当前预算。
    # resource_ready = ???  # 延迟、吞吐、显存三个目标至少达到一个。
    if not all(name in locals() for name in ("evidence_ready", "quality_ready", "resource_ready")):
        raise NotImplementedError("TODO 3：请完成三个部署状态判断")
    if not contract.get("ready"):
        return {"decision": "reject", "reason": "配对契约不一致"}
    if not evidence_ready:
        return {"decision": "pending", "reason": "真实量化执行证据尚未完整"}
    if not quality_ready:
        return {"decision": "reject", "reason": "候选评测质量退化超过预算"}
    if not resource_ready:
        return {"decision": "tune", "reason": "当前工作负载尚未达到资源或性能门槛"}
    return {"decision": "accept", "reason": "固定工作负载下质量、执行路径和资源门槛均成立"}


## 测试

分别核对来源与固定条件、指标换算和状态优先级。


In [ ]:
def _pair_records():
    """构造同模型、同负载和同环境的门禁、G0 与 G1 记录。"""
    gate = {
        "decision": "accept",
        "artifact_id": "qwen-awq-r1",
        "model_revision": "abc123",
    }
    fixed = {
        "model_revision": "abc123", "tokenizer_revision": "abc123",
        "workload_id": "fixed-chat-v1", "backend": "vllm-0.11",
        "hardware_id": "gpu-a", "cache_policy": "default",
    }
    baseline = {
        **fixed, "artifact_id": "qwen-float-r1", "tpot_ms": 20.0,
        "throughput_tps": 50.0, "peak_memory_mb": 8000.0, "quality": 10.0,
    }
    candidate = {
        **fixed, "artifact_id": "qwen-awq-r1", "tpot_ms": 18.0,
        "throughput_tps": 56.0, "peak_memory_mb": 5600.0, "quality": 10.4,
    }
    return gate, baseline, candidate


def test_deployment_pair_contract():
    """验证门禁、产物身份和固定字段漂移会被分别识别。"""
    gate, baseline, candidate = _pair_records()
    assert validate_deployment_pair(gate, baseline, candidate)["ready"]
    drifted = {**candidate, "hardware_id": "gpu-b"}
    result = validate_deployment_pair(gate, baseline, drifted)
    assert not result["ready"] and result["fixed_mismatches"] == ["hardware_id"]
    wrong_lineage = {**gate, "model_revision": "other-revision"}
    result = validate_deployment_pair(wrong_lineage, baseline, candidate)
    assert not result["ready"] and "model_revision_mismatch" in result["issues"]
    missing = {**baseline, 'workload_id': None}
    assert not validate_deployment_pair(gate, missing, {**candidate, 'workload_id': None})['ready']


def test_paired_metric_ledger():
    """验证延迟、吞吐、显存和质量使用正确方向，缺失显存保持为空。"""
    _, baseline, candidate = _pair_records()
    summary = summarize_deployment_pair(baseline, candidate)
    assert summary == {
        "tpot_ratio": 0.9, "throughput_ratio": 1.12,
        "memory_reduction_ratio": 0.3, "quality_ratio": 1.04,
    }
    assert summarize_deployment_pair(baseline, {**candidate, "peak_memory_mb": None})["memory_reduction_ratio"] is None
    for value in (float('nan'), float('inf'), True):
        try:
            _positive_ratio(value, 1.0)
        except ValueError:
            continue
        raise AssertionError('非法指标不应进入部署比值')


def test_deployment_state_machine():
    """验证缺证据、质量失败、待调优和接受四条项目路径。"""
    gate, baseline, candidate = _pair_records()
    contract = validate_deployment_pair(gate, baseline, candidate)
    comparison = summarize_deployment_pair(baseline, candidate)
    real_evidence = {"real_backend": True, "quantized_path": True, "fallback": False}
    assert decide_quantized_deployment(contract, comparison, real_evidence)["decision"] == "accept"
    assert decide_quantized_deployment(contract, comparison, {**real_evidence, "real_backend": False})["decision"] == "pending"
    assert decide_quantized_deployment(contract, {**comparison, "quality_ratio": 1.3}, real_evidence)["decision"] == "reject"
    weak = {**comparison, "tpot_ratio": 1.4, "throughput_ratio": 0.7, "memory_reduction_ratio": 0.05}
    assert decide_quantized_deployment(contract, weak, real_evidence)["decision"] == "tune"


def run_quantized_deployment_tests():
    """按项目执行顺序运行配对契约、指标账本与状态机测试。"""
    test_deployment_pair_contract()
    test_paired_metric_ledger()
    test_deployment_state_machine()
    print("✅ 配对契约、指标方向和部署状态机均已验证。")


run_quantized_deployment_tests()


---

## 参考代码与解析


### 代码


In [ ]:
FIXED_PAIR_FIELDS = (
    "model_revision", "tokenizer_revision", "workload_id",
    "backend", "hardware_id", "cache_policy",
)


def _positive_ratio(candidate_value: float, baseline_value: float) -> float:
    """计算正数指标的候选/基线比值。"""
    if not _finite_number(baseline_value, positive=True) or not _finite_number(candidate_value) or candidate_value < 0:
        raise ValueError("指标必须满足 baseline > 0 且 candidate >= 0")
    return candidate_value / baseline_value


def validate_deployment_pair(
    gate: Dict[str, Any], baseline: Dict[str, Any], candidate: Dict[str, Any]
) -> Dict[str, Any]:
    """检查产物门禁与 G0/G1 是否构成可比较的部署配对。"""
    # 给定实现：提取门禁、产物血缘和固定字段漂移等原子事实。
    gate_ready = gate.get("decision") == "accept"
    artifact_matches = _text(candidate.get("artifact_id")) and candidate.get("artifact_id") == gate.get("artifact_id")
    model_revision_matches = _text(candidate.get("model_revision")) and candidate.get("model_revision") == gate.get("model_revision")
    fixed_mismatches = [
        field for field in FIXED_PAIR_FIELDS if not _text(baseline.get(field)) or baseline.get(field) != candidate.get(field)
    ]
    # TODO 1（项目机制：配对契约）：组合血缘门禁与固定条件。
    lineage_ready = gate_ready and artifact_matches and model_revision_matches
    fixed_fields_ready = not fixed_mismatches
    issues = []
    if not gate_ready:
        issues.append("artifact_gate_not_accepted")
    if not artifact_matches:
        issues.append("artifact_id_mismatch")
    if not model_revision_matches:
        issues.append("model_revision_mismatch")
    issues.extend(f"fixed_field_mismatch:{field}" for field in fixed_mismatches)
    return {
        "ready": lineage_ready and fixed_fields_ready,
        "issues": issues,
        "fixed_mismatches": fixed_mismatches,
    }


def summarize_deployment_pair(
    baseline: Dict[str, Any], candidate: Dict[str, Any]
) -> Dict[str, Any]:
    """把延迟、吞吐、显存和质量转换成方向明确的配对指标。"""
    # TODO 2（项目机制：指标账本）：比值小于 1 的 TPOT 更好，比值大于 1 的吞吐更好。
    tpot_ratio = _positive_ratio(float(candidate["tpot_ms"]), float(baseline["tpot_ms"]))
    throughput_ratio = _positive_ratio(float(candidate["throughput_tps"]), float(baseline["throughput_tps"]))
    if baseline.get("peak_memory_mb") is None or candidate.get("peak_memory_mb") is None:
        memory_reduction_ratio = None
    else:
        memory_reduction_ratio = 1.0 - _positive_ratio(float(candidate["peak_memory_mb"]), float(baseline["peak_memory_mb"]))
    quality_ratio = _positive_ratio(float(candidate["quality"]), float(baseline["quality"]))
    return {
        "tpot_ratio": round(tpot_ratio, 6),
        "throughput_ratio": round(throughput_ratio, 6),
        "memory_reduction_ratio": None if memory_reduction_ratio is None else round(memory_reduction_ratio, 6),
        "quality_ratio": round(quality_ratio, 6),
    }


def decide_quantized_deployment(
    contract: Dict[str, Any], comparison: Dict[str, Any], evidence: Dict[str, Any],
    *, max_tpot_ratio: float = 1.20, min_throughput_ratio: float = 0.90,
    min_memory_reduction_ratio: float = 0.20, max_quality_ratio: float = 1.10,
) -> Dict[str, str]:
    """根据配对结果和真实执行证据形成阶段化部署决策。"""
    if not all(_finite_number(v) and v >= 0 for v in (max_tpot_ratio, min_throughput_ratio, min_memory_reduction_ratio, max_quality_ratio)):
        raise ValueError('部署门槛必须是有限非负数')
    if any(not _finite_number(comparison.get(k)) or comparison[k] < 0 for k in ('tpot_ratio', 'throughput_ratio', 'quality_ratio')):
        return {'decision': 'pending', 'reason': '部署比较缺少有效数值'}
    memory_gain = comparison.get("memory_reduction_ratio")
    # 给定实现：从真实记录中提取执行路径与三类资源目标。
    backend_observed = evidence.get("real_backend") is True
    quantized_path_observed = evidence.get("quantized_path") is True
    no_fallback = evidence.get("fallback") is False
    latency_ready = comparison.get("tpot_ratio", float("inf")) <= max_tpot_ratio
    throughput_ready = comparison.get("throughput_ratio", 0.0) >= min_throughput_ratio
    memory_ready = memory_gain is not None and memory_gain >= min_memory_reduction_ratio
    # TODO 3（项目机制：部署状态机）：组合执行、质量和资源门槛。
    evidence_ready = backend_observed and quantized_path_observed and no_fallback
    quality_ready = comparison.get("quality_ratio", float("inf")) <= max_quality_ratio
    resource_ready = latency_ready or throughput_ready or memory_ready
    if not contract.get("ready"):
        return {"decision": "reject", "reason": "配对契约不一致"}
    if not evidence_ready:
        return {"decision": "pending", "reason": "真实量化执行证据尚未完整"}
    if not quality_ready:
        return {"decision": "reject", "reason": "候选评测质量退化超过预算"}
    if not resource_ready:
        return {"decision": "tune", "reason": "当前工作负载尚未达到资源或性能门槛"}
    return {"decision": "accept", "reason": "固定工作负载下质量、执行路径和资源门槛均成立"}


### 解析

#### TODO 1：核对来源与固定条件

- 产物需已被接受，候选产物和模型版本需与来源一致；两次运行的模型、分词器、请求、后端、硬件和缓存条件保持一致。
- 答案将来源检查与固定字段检查合并。测试改变硬件和模型版本，核对错误来源不会进入性能判断。

#### TODO 2：换算配对指标

- 延迟、吞吐与本题质量指标均取候选／基线比；显存取 1 减去该比值，得到相对降幅。
- 答案按方向计算并保留缺失显存为 `None`。测试核对数值和缺测，避免把缺少记录当作零占用。

#### TODO 3：形成部署状态

- 后端、量化路径和无回退是合取条件，质量单独判断；资源部分按给定规则，延迟、吞吐和显存满足任一项即可。
- 答案先处理不可比与缺证，再检查质量和资源。四种状态样例核对优先级；阈值允许的回退范围由配置决定。


## Step 5：比较浮点与量化部署

固定产物、模型和请求集后，交换两条路径的运行先后。默认适配器保存请求与日志，实际量化计算路径还需独立核实；路径尚未确认时保留 `pending`，已有启动参数可用于排查加载配置。

| 路径 | 核对对象 | 输出 |
| --- | --- | --- |
| CPU | 固定身份、指标方向和状态规则 | 配对与决策测试 |
| 真实后端 | 同版本模型，三轮交替先后的配对 | 请求、日志和运行身份 |
| 收尾 | 来源、文件摘要、调用配置与完整性 | 报告和回执；接受时生成部署引用 |


### 5.1 环境与运行配置

使用 [INF-VLLM-011-CU128](../topic_discussion/quantization_compression/quickstart.md#q-vllm-quickstart)。填写本次唯一标签、阶段和产物评估的结果标签；浮点模型版本与量化产物先准备在本地。默认 `prepare` 读取并检查固定合同。


In [ ]:
RUN_LABEL = 'replace_with_unique_label'  # 本次唯一标签；不覆盖既有结果。
RUN_STAGE = 'prepare'  # prepare / measure / close；默认只读预检。
UPSTREAM_RUN_LABEL = 'replace_with_upstream_label'  # 82准入已接受结果的唯一标签。


### 5.2 环境与唯一上游预检

从82准入引用读取来源，核对SHA256、接受状态、嵌入的候选清单、实际产物文件与固定请求集。只有首次measure会检查GPU环境；prepare、close和已有测量的读取不初始化GPU。


In [ ]:
import importlib
import json

from tools.project_runtime import resolve_project_root
import tools.quantized_deployment_runtime as deployment_runtime

deployment_runtime = importlib.reload(deployment_runtime)
plan = deployment_runtime.project_plan(resolve_project_root(), '67', RUN_LABEL, UPSTREAM_RUN_LABEL, RUN_STAGE)
prepared = deployment_runtime.preflight_project(plan)
print({'ready': prepared['ready'], 'issues': prepared['issues'],
       'profile': plan['contract']['environment_profile'], 'inputs': prepared.get('snapshot')})


### 5.3 检查配对条件与门槛

固定模型与分词器版本、后端、请求、缓存和硬件，改变权重产物及其量化加载路径。三轮先后为浮点→量化、量化→浮点、浮点→量化。当前运行器使用每组 TPOT 的 P50 计算配对比值，再取三轮比值的中位数；质量沿用上游候选选择评测。当前服务适配器未提供可用峰值，显存降幅保留为空。


In [ ]:
print({'fixed_contract': plan['contract'],
       'artifact': prepared.get('artifact'),
       'quality_source': '82_candidate_selection_evaluation'})


### 5.4 真实执行并保存原始证据

measure按固定矩阵执行，逐项保存来源身份、原始请求结果、完整日志和失败。完整成员可在中断后复用；后端输出已存在但成员记录未完成时保留现场并使用新标签，避免覆盖。此处不生成稳定引用。


In [ ]:
if RUN_STAGE == 'measure':
    if not prepared['ready']:
        raise RuntimeError('预检未通过：' + '; '.join(prepared['issues']))
    measured = deployment_runtime.measure_project(plan)
    print({'status': measured['status'], 'decision': measured['decision']})
else:
    print({'status': 'skipped', 'stage': RUN_STAGE})


### 5.5 只读核对已保存证据

重新定位本标签的记录，逐项检查来源、固定合同、成员数量、身份、实际请求数及文件哈希，再重算决策。已有报告、交接和回执必须与原始证据一致；读取不会启动后端或补写结果。


In [ ]:
if prepared['ready']:
    checked = deployment_runtime.read_project(plan)
else:
    checked = {'status': 'blocked', 'decision': {'decision': 'pending'}, 'issues': prepared['issues']}
print({'status': checked['status'], 'decision': checked['decision'],
       'profiles': checked.get('report', {}).get('profile_decisions'),
       'issues': checked.get('issues', [])})


### 5.6 形成决策、登记与交接

close只消费已保存证据，生成报告和SHA256回执；内容一致时复用，中断时补齐缺失资产，冲突时停止。仅accept生成67部署引用，交给83。

| 已核对证据 | 决策 | 下一动作 |
|---|---|---|
| 缺少测量、请求失败或量化执行未核实 | pending | 补齐证据；失败复测使用新标签 |
| 配对身份不一致或候选质量退化超限 | reject | 保留报告，修复比较条件 |
| 质量与执行有效但当前性能门槛未达 | tune | 调整后用新标签复测 |
| 质量、实际执行与至少一项资源门槛通过 | accept | 生成引用，进入83压力实验 |

**当前登记结果**：尚无本页新协议的真实 GPU 结果。完成固定协议的真实测量后登记标签与结论；当前状态见[实验资产表](../topic_discussion/quantization_compression/experiment_assets.md)。


In [ ]:
if RUN_STAGE == 'close':
    closed = deployment_runtime.close_project(plan) if prepared['ready'] else checked
    print(closed)
else:
    print({'status': 'skipped', 'stage': RUN_STAGE})


## 相关阅读

完成固定请求比较后，再检查压力增加时的延迟与吞吐，并对照后端支持范围。

- [量化服务配置基准](./83_Quantized_Serving_Profile_Benchmark.ipynb)：在三类请求压力下检查服务目标。
- [vLLM 量化文档](https://docs.vllm.ai/en/latest/features/quantization/)：核对格式、硬件与执行实现的兼容范围。
